# Phase 5 prime bundle-first n=10 controls (Colab)

This notebook runs the existing `experiments/44_phase5_prime_bundle_first.py` harness on Colab. It is a Phase 5 prime diagnostic, not a Phase 5 graduation run.

The default path avoids Google Drive mount and writes results under `/content`. Download result JSONs from the final cell if you want to preserve them.

Planned sequence:

1. Clone `Dypatterson/Neuro-AI` and check out `phase5-m1-role-energy-stack`.
2. Run a tiny smoke.
3. Run the hard-cell n=10 control rerun in parallel shards: `D=4096`, `N=512`, `K_roles=16`, `cue_noise=0.15`, `scene_token=1`, skewed co-occurrence.
4. Run a candidate-only n=10 grid in parallel shards across K, N, cue noise, scene-token, and co-occurrence.
5. Keep the full all-controls matrix as an explicit opt-in parallel-sharded cell because it is much larger.

Default parallelism is `MAX_PARALLEL = 8`; lower it if Colab becomes unstable, or raise it if GPU memory remains low.


In [ ]:
# 1. Clone the repo and check out the active branch.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git /content/Neuro-AI
%cd /content/Neuro-AI
!git checkout phase5-m1-role-energy-stack
!git log --oneline -1

# Verify the Phase 5 prime harness exists on this branch.
!test -f experiments/44_phase5_prime_bundle_first.py
!grep -n "bundle-first structural-memory" experiments/44_phase5_prime_bundle_first.py | head -1

In [ ]:
# 2. Runtime, harness sanity, and parallel shard helpers.
import json, os, subprocess, time
from pathlib import Path

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("device", DEVICE)

if DEVICE == "cuda":
    !nvidia-smi --query-gpu=name,memory.total --format=csv

!PYTHONPATH=src:. python -m py_compile experiments/44_phase5_prime_bundle_first.py

MAX_PARALLEL = 8  # Raise/lower based on Colab GPU memory and process stability.
HARNESS = "experiments/44_phase5_prime_bundle_first.py"
ALL_CONDITIONS = [
    "candidate",
    "random_role",
    "shuffled_role",
    "perfect_cue",
    "bundle_positive",
    "content_cleanup_positive",
]

def merge_payloads(part_paths, merged_out, label):
    payloads = [json.loads(Path(p).read_text()) for p in part_paths]
    merged = {
        "framing": payloads[0].get("framing", {}),
        "config": {
            "label": label,
            "parallel_shards": [str(p) for p in part_paths],
            "merged_by_notebook": True,
        },
        "aggregates": {},
        "raw": [],
    }
    for payload in payloads:
        for key, value in payload.get("aggregates", {}).items():
            if key in merged["aggregates"]:
                raise ValueError(f"duplicate aggregate key while merging: {key}")
            merged["aggregates"][key] = value
        merged["raw"].extend(payload.get("raw", []))
    Path(merged_out).write_text(json.dumps(merged, indent=2))
    print(f"merged {len(part_paths)} shards -> {merged_out}")
    return merged

def _launch_shard(name, args, out_root):
    out_path = out_root / f"{name}.json"
    log_path = out_root / f"{name}.log"
    cmd = ["python", HARNESS, *map(str, args), "--out", str(out_path)]
    env = dict(os.environ, PYTHONPATH="src:.")
    logf = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
    return {"name": name, "proc": proc, "logf": logf, "out": out_path, "log": log_path, "cmd": cmd}

def run_parallel_shards(label, shards, merged_out, max_parallel=MAX_PARALLEL, poll_s=20):
    out_root = Path(merged_out).with_suffix("")
    out_root.mkdir(parents=True, exist_ok=True)
    pending = list(shards)
    running = []
    completed = []
    failed = []
    t0 = time.time()
    print(f"{label}: {len(pending)} shards, max_parallel={max_parallel}, device={DEVICE}")
    while pending or running:
        while pending and len(running) < max_parallel:
            name, args = pending.pop(0)
            shard = _launch_shard(name, args, out_root)
            running.append(shard)
            print(f"  launched {name}")
        time.sleep(poll_s)
        still = []
        for shard in running:
            code = shard["proc"].poll()
            if code is None:
                still.append(shard)
                continue
            shard["logf"].close()
            elapsed = (time.time() - t0) / 60.0
            if code == 0 and shard["out"].exists():
                completed.append(shard)
                print(f"  done {shard['name']} at {elapsed:.1f} min")
            else:
                failed.append((shard, code))
                print(f"  FAILED {shard['name']} code={code} at {elapsed:.1f} min")
        running = still
        if DEVICE == "cuda":
            subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader"], check=False)
    if failed:
        for shard, code in failed:
            print(f"\n--- tail {shard['log']} ---")
            lines = Path(shard["log"]).read_text(errors="replace").splitlines()
            print("\n".join(lines[-80:]))
        raise RuntimeError(f"{label}: {len(failed)} shard(s) failed")
    return merge_payloads([s["out"] for s in completed], merged_out, label)


In [ ]:
# 3. Tiny smoke. Expected: candidate > controls, positives at 1.0 on this small cell.
!PYTHONPATH=src:. python experiments/44_phase5_prime_bundle_first.py \
  --Ds 128 --Ns 8 --K_roles 2 --cue_noise 0.0 \
  --seeds 17 --n_queries 8 --C_codebook 32 \
  --conditions candidate random_role shuffled_role perfect_cue bundle_positive content_cleanup_positive \
  --scene_token 0 --cooccurrence uniform --device cpu \
  --out /content/phase5_prime_smoke.json

In [ ]:
# 4. Hard-cell n=10 controls, sharded by condition.
# This reproduces the most important Report 069 cell while running controls concurrently.
HARD_CELL_OUT = "/content/phase5_prime_bundle_hard_cell_n10.json"
HARD_CONDITIONS = ALL_CONDITIONS
hard_shards = []
for condition in HARD_CONDITIONS:
    args = [
        "--Ds", 4096,
        "--Ns", 512,
        "--K_roles", 16,
        "--cue_noise", 0.15,
        "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
        "--n_queries", 512,
        "--C_codebook", 1024,
        "--conditions", condition,
        "--scene_token", 1,
        "--cooccurrence", "skewed",
        "--device", DEVICE,
    ]
    hard_shards.append((f"hard_{condition}", args))

hard_payload = run_parallel_shards("hard-cell-controls", hard_shards, HARD_CELL_OUT)

In [ ]:
# 5. Summarize the hard-cell controls.
def summarize_payload(path):
    payload = json.loads(Path(path).read_text())
    rows = []
    for key, agg in payload["aggregates"].items():
        rows.append((key, agg))
    for key, agg in sorted(rows):
        print(
            f"{key}\n"
            f"  top1={agg['top1_mean']:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] "
            f"scene_tix={agg['scene_tix_rate']:.4f} content_tix={agg['content_tix_rate']:.4f} "
            f"n={agg['n_total']}"
        )
    return payload

hard_payload = summarize_payload(HARD_CELL_OUT)

In [ ]:
# 6. Candidate-only n=10 grid, sharded by K x scene-token x co-occurrence.
# This launches 16 independent shards and keeps up to MAX_PARALLEL active at once.
CANDIDATE_GRID_OUT = "/content/phase5_prime_candidate_grid_n10.json"
candidate_shards = []
for K in [2, 4, 8, 16]:
    for scene_token in [0, 1]:
        for cooc in ["uniform", "skewed"]:
            args = [
                "--Ds", 4096,
                "--Ns", 16, 32, 64, 128, 256, 512,
                "--K_roles", K,
                "--cue_noise", 0.0, 0.05, 0.10, 0.15,
                "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
                "--n_queries", 256,
                "--C_codebook", 1024,
                "--conditions", "candidate",
                "--scene_token", scene_token,
                "--cooccurrence", cooc,
                "--device", DEVICE,
            ]
            candidate_shards.append((f"candidate_K{K}_scene{scene_token}_{cooc}", args))

candidate_payload = run_parallel_shards("candidate-grid", candidate_shards, CANDIDATE_GRID_OUT)

In [ ]:
# 7. Summarize candidate grid: worst and best cells by top1.
candidate_payload = json.loads(Path(CANDIDATE_GRID_OUT).read_text())
rows = sorted(
    ((agg["top1_mean"], key, agg) for key, agg in candidate_payload["aggregates"].items()),
    key=lambda x: x[0],
)

print("Worst 20 candidate cells:")
for top1, key, agg in rows[:20]:
    print(f"{top1:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}")

print("\nBest 20 candidate cells:")
for top1, key, agg in rows[-20:]:
    print(f"{top1:.4f} CI=[{agg['wilson_lo']:.4f},{agg['wilson_hi']:.4f}] scene={agg['scene_tix_rate']:.4f} content={agg['content_tix_rate']:.4f} :: {key}")

In [ ]:
# 8. Optional full all-controls matrix, sharded by condition x K x scene-token x co-occurrence.
# This creates 96 shards. Keep MAX_PARALLEL conservative if Colab becomes unstable.
RUN_FULL_MATRIX = False

if RUN_FULL_MATRIX:
    FULL_MATRIX_OUT = "/content/phase5_prime_full_controls_grid_n10.json"
    full_shards = []
    for condition in ALL_CONDITIONS:
        for K in [2, 4, 8, 16]:
            for scene_token in [0, 1]:
                for cooc in ["uniform", "skewed"]:
                    args = [
                        "--Ds", 4096,
                        "--Ns", 16, 32, 64, 128, 256, 512,
                        "--K_roles", K,
                        "--cue_noise", 0.0, 0.05, 0.10, 0.15,
                        "--seeds", 17, 11, 23, 1, 2, 3, 5, 7, 13, 29,
                        "--n_queries", 512,
                        "--C_codebook", 1024,
                        "--conditions", condition,
                        "--scene_token", scene_token,
                        "--cooccurrence", cooc,
                        "--device", DEVICE,
                    ]
                    full_shards.append((f"full_{condition}_K{K}_scene{scene_token}_{cooc}", args))
    full_payload = run_parallel_shards("full-controls-grid", full_shards, FULL_MATRIX_OUT)
else:
    print("Full all-controls matrix skipped. Set RUN_FULL_MATRIX = True to run it.")

In [ ]:
# 9. Download result JSONs from the Colab runtime.
from google.colab import files

for path in [
    "/content/phase5_prime_smoke.json",
    "/content/phase5_prime_bundle_hard_cell_n10.json",
    "/content/phase5_prime_candidate_grid_n10.json",
    "/content/phase5_prime_full_controls_grid_n10.json",
]:
    if Path(path).exists():
        print("downloading", path)
        files.download(path)
